In [1]:
#モデルの読み込みと、以前エラーになった Colab 判定を修正した安全な初期化コードです。
# 既存の .ckpt ファイルを直接ロードします。

import sys
from pathlib import Path
import numpy as np
import torch
import ase
from ase.io import read, write
from ase.build import molecule, bulk, fcc111, bcc110, add_adsorbate
from ase.cluster import Icosahedron, Octahedron
import py3Dmol

from torch.utils.data import DataLoader
from nequip.data import AtomicDataDict
from nequip.data.dataset import ASEDataset
from nequip.data.transforms import NeighborListTransform, ChemicalSpeciesToAtomTypeMapper
from nequip.scripts.compile import load_saved_model, _EAGER_MODEL_KEY, _SOLE_MODEL_KEY
from nequip.nn import graph_model
from nequip.nn.pair_potential import ZBL
from nequip.utils.global_state import set_global_state

# デバイスとパス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKDIR = Path.cwd()
CKPT_PATH = WORKDIR / "s128_t64.nequip.ckpt"

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {CKPT_PATH}")

# モデルハイパーパラメータ取得
checkpoint = torch.load(str(CKPT_PATH), map_location=device, weights_only=False)
inner_model_config = checkpoint["hyper_parameters"]["model"].get("model", {})

avgn = inner_model_config.get("avg_num_neighbors", 1.0)
sgij = 1.0 / np.sqrt(2.0 * avgn)
eshft = inner_model_config.get("per_type_energy_shifts", 0.0)
escal = inner_model_config.get("per_type_energy_scales", 1.0)

set_global_state(allow_tf32=True)

# モデル本体のロード
model = load_saved_model(str(CKPT_PATH), _EAGER_MODEL_KEY, _SOLE_MODEL_KEY).to(device)
model.eval()

type_names = model.metadata[graph_model.TYPE_NAMES_KEY].split(" ")
r_max = float(model.metadata[graph_model.R_MAX_KEY])

# ZBLモジュールの準備
zbl_module = ZBL(
    type_names,
    type_names,
    units="metal",
    irreps_in={AtomicDataDict.NORM_LENGTH_KEY: "0e"},
).to(device)

print(f"✅ Ready: Loaded model on {device} (r_max = {r_max} Å, Supported Elements = {len(type_names)})")

/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-10-04 15:12:11,646][nequip.model.saved_models.load_utils][INFO] - [rank: 0] Loading model from /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...
[2026-10-04 15:12:11,648][nequip.model.saved_models.checkpoint][INFO] - [rank: 0] Loading model from checkpoint file: /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...


/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `torch` versions differ between the checkpoint file (2.7.1) and the current run (2.14.1+cpu) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `e3nn` versions differ between the checkpoint file (0.5.6) and the current run (0.6.0) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `nequip` versions differ between the checkpoint file (0.14.0) and the current run (0.19.1) -- `ModelFromCheckpoint` will be built with the current run's v

[2026-10-04 15:12:12,892][nequip.train.ema][INFO] - [rank: 0] Loading EMA weights for evaluation model.
✅ Ready: Loaded model on cpu (r_max = 6.0 Å, Supported Elements = 89)


In [2]:
#入力された構造（分子名・ASE オブジェクト・外部ファイル）から自動で $D_{ij}$ を計算し、表と 3D 図を出す関数群です。

def make_dataloader_for_xyz(xyz_path, chemical_symbols, r_max, device):
    """XYZファイルから推論用DataLoaderを作成"""
    transforms = [
        NeighborListTransform(r_max=r_max),
        ChemicalSpeciesToAtomTypeMapper(chemical_symbols),
    ]
    dataset = ASEDataset(str(xyz_path), transforms=transforms)
    dl = DataLoader(dataset, batch_size=1, collate_fn=AtomicDataDict.batched_from_list)

    class DeviceDataLoader:
        def __init__(self, dl, device):
            self.dl = dl
            self.device = device
        def __iter__(self):
            for batch in self.dl:
                yield {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        def __len__(self):
            return len(self.dl)

    return DeviceDataLoader(dl, device)


def calculate_dij_table(out, chemical_symbols, cutoff=5.0):
    """モデルの出力テンソルからペアごとの D_ij テーブルを集計して表示"""
    DKEY = AtomicDataDict
    edge_eng = out[DKEY.EDGE_ENERGY_KEY].squeeze().cpu().detach().numpy()
    edge_idx = out[DKEY.EDGE_INDEX_KEY].squeeze().cpu().detach().numpy().T
    edge_r = out[DKEY.EDGE_LENGTH_KEY].squeeze().cpu().detach().numpy()
    types = out[DKEY.ATOM_TYPE_KEY].squeeze().cpu().detach().numpy()

    # ZBL項の計算
    try:
        zbl_out = zbl_module._zbl(
            Z=zbl_module.atomic_numbers,
            r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
            atom_types=out[DKEY.ATOM_TYPE_KEY],
            edge_index=out[DKEY.EDGE_INDEX_KEY],
            qqr2exesquare=zbl_module._qqr2exesquare,
        )
        zbl_cutoff = zbl_module.cutoff(out[DKEY.NORM_LENGTH_KEY]).to(torch.get_default_dtype())
        edge_eng_zbl = (zbl_out * zbl_cutoff).squeeze().cpu().detach().numpy()
    except Exception:
        edge_eng_zbl = np.zeros_like(edge_eng)

    print(f"\n{'idx_i':>5} {'idx_j':>5}  {'Pair':^7} {'r (Å)':^8} {'D_ij (eV)':^10}")
    print("-" * 45)

    rec = {}
    cnt, sum_dij = 0, 0.0
    for e_idx, eng, eng_z, r in zip(edge_idx, edge_eng, edge_eng_zbl, edge_r):
        r_val = float(r)
        if r_val > cutoff:
            continue
        i, j = int(e_idx[0]), int(e_idx[1])
        key = (i, j, round(r_val, 6))
        rev = (j, i, round(r_val, 6))

        if rev in rec:
            other = rec.pop(rev)
            dij_core = float((eng + other["eng"]) * sgij * escal)
            dij_total = dij_core + 2.0 * float(eng_z)
            sum_dij += dij_total
            cnt += 1
            sym_i, sym_j = chemical_symbols[int(types[i])], chemical_symbols[int(types[j])]
            print(f"{i:5d} {j:5d}  {sym_i}-{sym_j:<2s}   {r_val:8.3f} {dij_total:10.4f}")
        else:
            rec[key] = {"eng": float(eng), "eng_zbl": float(eng_z)}

    print("-" * 45)
    print(f"Pairs: {cnt} | Total D_ij Sum: {sum_dij:.4f} eV\n")


def inspect_and_analyze(structure_input, name="sample", cutoff=None, show_3d=True):
    """統合エントリーポイント"""
    cutoff = cutoff or r_max
    if isinstance(structure_input, (str, Path)):
        atoms = read(str(structure_input))
        temp_xyz = WORKDIR / f"temp_{Path(structure_input).stem}.xyz"
    else:
        atoms = structure_input
        temp_xyz = WORKDIR / f"temp_{name}.xyz"

    write(str(temp_xyz), atoms)

    if show_3d:
        with open(temp_xyz) as f:
            xyz_str = f.read()
        view = py3Dmol.view(width=600, height=350)
        view.addModel(xyz_str, "xyz")
        view.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.28}})
        for idx, (sym, pos) in enumerate(zip(atoms.get_chemical_symbols(), atoms.positions)):
            view.addLabel(
                f"{sym}{idx}",
                {"position": {"x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2])},
                 "fontSize": 11, "fontColor": "black", "backgroundColor": "white", "backgroundOpacity": 0.7, "inFront": True}
            )
        view.zoomTo()
        view.show()

    dl = make_dataloader_for_xyz(temp_xyz, type_names, r_max, device)
    for batch in dl:
        out = model(batch)
        print(f"Target: {name} (Formula: {atoms.get_chemical_formula()}, Atoms: {len(atoms)})")
        calculate_dij_table(out, type_names, cutoff=cutoff)
        return out, atoms

In [ ]:
# ==============================================================================
# 🧪 実験設定セル（分子 / 金属ナノ粒子 / 結晶構造ファイル）
# ==============================================================================
# 調べたい対象のブロックのコメント（#）を解除し、Shift + Enter で実行してください。
# ==============================================================================

# ------------------------------------------------------------------------------
# パターン 1: 有機分子・気体（ASE内蔵の162種など）
# ------------------------------------------------------------------------------
# 分子名例: 'H2O', 'CH4', 'CH3COOH', 'CH3CH2OH', 'benzene', 'glycine', 'CO2'
target_structure = molecule("CH3COOH")
target_name = "AceticAcid"
target_cutoff = None  # None の場合はデフォルトの 6.0 Å


# ------------------------------------------------------------------------------
# パターン 2: 金属ナノ粒子（ナノクラスター）
# ------------------------------------------------------------------------------
# ※調べたい場合は上のパターン1をコメントアウト（#）し、以下の行の # を外してください
# target_structure = Icosahedron("Pt", noshells=2)  # 13原子の白金クラスター
# target_name = "Pt13_Cluster"
# target_cutoff = 3.5  # 金属は第1近接（直接の結合）を捉えやすいよう短めに設定


# ------------------------------------------------------------------------------
# パターン 3: 外部の結晶ファイル（CIF / POSCAR / CONTCAR など）
# ------------------------------------------------------------------------------
# ※調べたい場合は上のパターンをコメントアウト（#）し、以下の行の # を外してください
# file_path = "TiO2.cif"         # プロジェクト直下に置いたファイル名
# repeat_dim = (2, 2, 2)         # (1, 1, 1) なら単位格子のまま、(2, 2, 2) なら拡張ブロック
# 
# crystal_raw = read(file_path)
# if repeat_dim != (1, 1, 1):
#     target_structure = crystal_raw.repeat(repeat_dim)
#     target_name = f"{Path(file_path).stem}_{repeat_dim[0]}x{repeat_dim[1]}x{repeat_dim[2]}"
# else:
#     target_structure = crystal_raw
#     target_name = Path(file_path).stem
# 
# target_cutoff = 3.5            # 結晶内部の配位結合を見る場合は 3.0〜3.5 Å が目安


# ==============================================================================
# 🚀 実行部（変更不要）
# ==============================================================================
out, atoms = inspect_and_analyze(
    target_structure, 
    name=target_name, 
    cutoff=target_cutoff
)

In [ ]:
# ==============================================================================
# 🔍 特定ペア解析セル（Selected Pair Inspection）
# ==============================================================================
# 上の3Dビューアーで確認した原子ラベル（例: C0, O1 など）の番号（0始まり）を指定します。
# ==============================================================================

# 調べたい2つの原子のインデックスを指定
atom_i = 0
atom_j = 1


# ------------------------------------------------------------------------------
# 🚀 実行部（変更不要）
# ------------------------------------------------------------------------------
if "out" not in globals() or "atoms" not in globals() or out is None:
    raise RuntimeError("直前の実験設定セルを実行して 'out' と 'atoms' を取得してください。")

n_atoms = len(atoms)

# 指定インデックスが有効範囲内かチェック
if not (0 <= atom_i < n_atoms and 0 <= atom_j < n_atoms):
    print(f"⚠️ 指定されたインデックス ({atom_i}, {atom_j}) は無効です。")
    print(f"   現在の構造 '{target_name}' の原子数は {n_atoms} 個（指定可能: 0 〜 {n_atoms - 1}）です。")
elif atom_i == atom_j:
    print(f"⚠️ 同一の原子 ({atom_i}, {atom_j}) が指定されています。異なる原子を指定してください。")
else:
    sym_i = atoms.get_chemical_symbols()[atom_i]
    sym_j = atoms.get_chemical_symbols()[atom_j]
    
    print("=" * 60)
    print(f" 🎯 対象ペア: {sym_i}{atom_i} - {sym_j}{atom_j}  (系: {target_name})")
    print("=" * 60)
    
    # チュートリアル由来の個別ペア詳細関数を実行
    show_pair_energy(
        out,
        atom_i=atom_i,
        atom_j=atom_j,
        chemical_symbols=type_names,
        zbl_module=zbl_module,
    )